# 04 Perceptron, MLP & Framework Setup | البيرسِبتْرون والـ MLP وإعداد الإطارات

## 📚 Learning Objectives | أهداف التعلم

By completing this notebook (~20 min), you will:
- Check TensorFlow and PyTorch setup and compare deep learning vs traditional ML
- Implement a simple perceptron from scratch and train it on a tiny task (e.g. AND gate)
- Build a small MLP in TensorFlow and in PyTorch so you see both frameworks

---

## 🌍 Real life | في الواقع

**Where is this used?** Perceptrons and MLPs are the **building blocks** of deep learning. TensorFlow and PyTorch are the two main frameworks used in **industry and research** for training neural networks.

**In this notebook we use** a **perceptron** (one neuron) and an **MLP** (multiple layers) to **see how frameworks define and run models**. We use **both TensorFlow and PyTorch** (instead of only one) **because** in real life you will meet both; knowing how to build a small MLP in each gets you ready for Units 2–5.

**📌 Covers slide(s):** **02** — ANNs, perceptron, MLP, activation, Keras; **06** — TensorFlow vs PyTorch, Colab, Jupyter. *Do this notebook after those slides.*

---

**Before starting:** Run the imports cell below. If TensorFlow or PyTorch fails, see `DOCS/COLAB_SETUP.md`.


## Theory (short) | النظرية

- **Perceptron** = one neuron: inputs × weights + bias → step function; can learn simple linear boundaries (e.g. AND gate).
- **MLP (Multi-Layer Perceptron)** = several layers of neurons (Dense/Linear); can learn non-linear patterns; this is what we used in 02_simple_neural_network.
- **TensorFlow/Keras**: high-level API (`Sequential`, `Dense`); easy to get started; common in industry.
- **PyTorch**: `nn.Module`, `nn.Linear`; dynamic graph; common in research. Both are used in real projects.
- **Data flow:** input → layer 1 → activation → layer 2 → … → output. Frameworks automate backprop and optimizer.



## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** NumPy, Matplotlib, TensorFlow (optional), PyTorch (optional). We use a tiny AND-gate dataset for the perceptron; no dataset for the MLP build (structure only).

**Outputs:** Printed framework versions (TF, PyTorch), short DL vs ML comparison, perceptron weights after training on AND gate, and small MLP model summaries (TF and PyTorch).


In [1]:
# Step 1: Imports and framework check
import numpy as np
import matplotlib.pyplot as plt

try:
    import tensorflow as tf
    HAS_TF = True
    print(f"✅ TensorFlow {tf.__version__}")
except Exception as e:
    err = str(e).lower()
    if "charset_normalizer" in err or "md__mypyc" in err or "partially initialized" in err:
        print("⚠️ Fix: pip install --upgrade charset-normalizer requests, then restart kernel.")
        raise RuntimeError("Fix: pip install --upgrade charset-normalizer requests, then restart kernel.") from e
    HAS_TF = False
    print("⚠️ TensorFlow not found. Install: pip install tensorflow")

try:
    import torch
    HAS_TORCH = True
    print(f"✅ PyTorch {torch.__version__}")
except ImportError:
    HAS_TORCH = False
    print("⚠️ PyTorch not found. Install: pip install torch")

print("✅ NumPy and Matplotlib ready.")


✅ TensorFlow 2.20.0


✅ PyTorch 2.9.1
✅ NumPy and Matplotlib ready.


## Step 2: Deep Learning vs Traditional ML (short)


In [2]:
# We use this comparison so students see why we use DL (automatic features) instead of manual feature engineering.
print("DL vs Traditional ML (short):")
print("  Features: DL learns from raw data; traditional ML often needs hand-designed features.")
print("  Data: DL benefits from more data; traditional ML can work with less.")
print("  Use: DL excels at images, text, audio; traditional ML is strong on tabular/structured data.")
print("✅ Deep learning automatically learns features — that's why we use it for complex patterns.")


DL vs Traditional ML (short):
  Features: DL learns from raw data; traditional ML often needs hand-designed features.
  Data: DL benefits from more data; traditional ML can work with less.
  Use: DL excels at images, text, audio; traditional ML is strong on tabular/structured data.
✅ Deep learning automatically learns features — that's why we use it for complex patterns.


## Step 3: Perceptron from scratch (we use it instead of an MLP here because we want to see one neuron learn a simple rule)


In [3]:
# Perceptron = one neuron: out = step(w·x + b). We use it to learn AND gate (instead of hand-coded AND) to see weights learned from data.
class Perceptron:
    def __init__(self, n_inputs, lr=0.1):
        self.weights = np.zeros(n_inputs)
        self.bias = 0.0
        self.lr = lr

    def step(self, x):
        return 1 if x >= 0 else 0

    def forward(self, x):
        return self.step(np.dot(x, self.weights) + self.bias)

    def fit(self, X, y, epochs=10):
        for _ in range(epochs):
            for xi, yi in zip(X, y):
                pred = self.forward(xi)
                err = yi - pred
                self.weights += self.lr * err * xi
                self.bias += self.lr * err

# AND gate: (0,0)->0, (0,1)->0, (1,0)->0, (1,1)->1
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y_and = np.array([0, 0, 0, 1])
p = Perceptron(n_inputs=2)
p.fit(X_and, y_and)
print("Perceptron weights after AND gate:", p.weights)
print("Bias:", p.bias)
print("Predictions:", [p.forward(x) for x in X_and])
print("Expected:   ", y_and.tolist())


Perceptron weights after AND gate: [0.2 0.1]
Bias: -0.20000000000000004
Predictions: [0, 0, 0, 1]
Expected:    [0, 0, 0, 1]


## Step 4: MLP with TensorFlow (we use Keras Sequential instead of raw TF so we build quickly)


In [4]:
if HAS_TF:
    # Small MLP: 4 inputs -> 8 hidden -> 2 outputs (e.g. binary). We use Dense layers (not conv) because input is flat.
    model_tf = tf.keras.Sequential([
        tf.keras.layers.Dense(8, activation="relu", input_shape=(4,)),
        tf.keras.layers.Dense(2, activation="softmax"),
    ])
    model_tf.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print("TensorFlow MLP summary:")
    model_tf.summary()
else:
    print("Skipping TF MLP (TensorFlow not available).")


TensorFlow MLP summary:


/opt/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            18 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58 (232.00 B)

 Trainable params: 58 (232.00 B)

 Non-trainable params: 0 (0.00 B)

## Step 5: MLP with PyTorch (same idea as TF: Linear layers; we use nn.Sequential for quick build)


In [5]:
if HAS_TORCH:
    # Same structure as TF: 4 -> 8 -> 2. We use nn.Linear (PyTorch) instead of Dense (Keras); same idea.
    model_pt = torch.nn.Sequential(
        torch.nn.Linear(4, 8),
        torch.nn.ReLU(),
        torch.nn.Linear(8, 2),
    )
    print("PyTorch MLP:")
    print(model_pt)
else:
    print("Skipping PyTorch MLP (PyTorch not available).")


PyTorch MLP:
Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=2, bias=True)
)


## 🧩 Mini-exercise | تمرين مصغر

**Try it:** In a new code cell, print the version of TensorFlow and of PyTorch (e.g. `tf.__version__` and `torch.__version__`). If one of them failed to import, note which one and what you would do to fix it.

---

## ✅ Summary | الملخص

**What you did:**
- Checked TensorFlow and PyTorch setup.
- Compared deep learning vs traditional ML in one line each.
- Implemented a perceptron from scratch and trained it on the AND gate (weights learned from data).
- Built a small MLP in TensorFlow (Keras Sequential) and in PyTorch (nn.Sequential).

**In real life you'd also:** Train the MLP on real data (e.g. MNIST), tune learning rate and layer sizes, and choose one framework per project.

**The main idea:** A perceptron is one neuron; an MLP is many layers. TensorFlow and PyTorch both define layers and run backprop; knowing how to build a small MLP in each prepares you for the rest of the course.

**Next:** `07_image_processing_feature_extraction` covers image basics and feature extraction before we use CNNs.
